# 🃏 Poker AI Championship Pipeline - Inference

**End-to-End Gameplay: Screenshot → Game State → Optimal Action**

This notebook runs the complete poker AI pipeline:
1. Load fine-tuned QWEN 2.5-7B VL for perception
2. Load DeepStack value network for decision
3. Process screenshots and output Nash equilibrium actions

## Requirements
- Google Colab with GPU (T4 recommended)
- Models saved to Google Drive
- Runtime Type: GPU

## 1️⃣ Setup & Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION - Edit these paths to match your Drive setup
# ═══════════════════════════════════════════════════════════════

CONFIG = {
    # Model paths on Google Drive
    "perception_model": "/content/drive/MyDrive/poker_ai/models/qwen_poker_vl",
    "decision_model": "/content/drive/MyDrive/poker_ai/models/deepstack_champion.pt",
    
    # Solver settings
    "cfr_iterations": 2000,
    "simulation_mode": True,  # Set False to enable GUI actions
    
    # Device settings
    "device": "auto",  # 'auto', 'cuda', or 'cpu'
    "use_quantization": True,  # 4-bit for memory efficiency
}

print("✅ Configuration loaded")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes peft trl
!pip install -q pydantic numpy torch

# Install Unsloth for efficient inference
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("✅ Dependencies installed")

In [ ]:
# Verify GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️ No GPU detected. Enable GPU in Runtime settings.")

## 2️⃣ Load Pipeline

In [ ]:
import sys
import os

# Add pipeline to path
sys.path.insert(0, '/content/drive/MyDrive/poker_ai/poker_ai_colab_pipeline')

# Import orchestrator
from src.orchestration import PokerWorkflowOrchestrator

print("✅ Pipeline modules loaded")

In [ ]:
# Initialize orchestrator
orchestrator = PokerWorkflowOrchestrator(
    perception_model=CONFIG["perception_model"],
    decision_model=CONFIG["decision_model"],
    simulation_mode=CONFIG["simulation_mode"],
    cfr_iterations=CONFIG["cfr_iterations"],
    device=CONFIG["device"]
)

print("✅ Orchestrator initialized")

## 3️⃣ Run Inference

In [ ]:
# Upload a screenshot for testing
from google.colab import files
from PIL import Image
import io

print("Upload a poker screenshot:")
uploaded = files.upload()

if uploaded:
    screenshot_path = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {screenshot_path}")
    
    # Display the image
    img = Image.open(screenshot_path)
    display(img.resize((600, 400)))

In [ ]:
# Process screenshot and get action
if 'screenshot_path' in dir() and screenshot_path:
    print("Processing screenshot...")
    result = orchestrator.process_screenshot(screenshot_path)
    
    print("\n" + "="*50)
    print("🎯 RECOMMENDED ACTION")
    print("="*50)
    print(f"Action: {result.action.upper()}")
    if result.amount > 0:
        print(f"Amount: {result.amount}")
    print(f"Confidence: {result.confidence:.1%}")
    print(f"Latency: {result.latency_ms:.0f}ms")
    
    if result.strategy:
        print("\nStrategy Breakdown:")
        for action, prob in result.strategy.items():
            print(f"  {action}: {prob:.1%}")
    
    print("\nReasoning:")
    print(f"  {result.reasoning}")
else:
    print("⚠️ No screenshot uploaded. Upload one above.")

## 4️⃣ Test with Manual Game State

In [ ]:
# Test solver directly with a game state
test_state = {
    'hole_cards': ['As', 'Kh'],  # Ace-King suited
    'community_cards': ['Qd', 'Jc', 'Ts'],  # Broadway draw
    'pot_size': 150,
    'current_bet': 50,
    'player_stack': 1000,
    'opponent_stack': 1200,
    'street': 'flop'
}

print("Testing with game state:")
print(f"  Hand: {test_state['hole_cards']}")
print(f"  Board: {test_state['community_cards']}")
print(f"  Pot: {test_state['pot_size']}")
print(f"  To Call: {test_state['current_bet']}")

result = orchestrator.solve_state(test_state)

print("\n" + "="*50)
print("🎯 SOLVER RESULT")
print("="*50)
print(f"Action: {result.action.upper()}")
if result.amount > 0:
    print(f"Amount: {result.amount}")
print(f"Confidence: {result.confidence:.1%}")
print(f"Latency: {result.latency_ms:.0f}ms")

## 5️⃣ Batch Processing

In [ ]:
# Process multiple game states
test_hands = [
    {'hole_cards': ['Ah', 'Ac'], 'pot_size': 100, 'current_bet': 20, 'street': 'preflop'},
    {'hole_cards': ['7s', '2d'], 'pot_size': 100, 'current_bet': 20, 'street': 'preflop'},
    {'hole_cards': ['Ks', 'Qs'], 'pot_size': 200, 'current_bet': 50, 'street': 'flop', 
     'community_cards': ['Js', 'Ts', '2h']},
]

print("Batch Processing Results")
print("="*60)

for i, state in enumerate(test_hands):
    result = orchestrator.solve_state(state)
    hand_str = ' '.join(state['hole_cards'])
    print(f"\nHand {i+1}: {hand_str} | Street: {state['street']}")
    print(f"  → {result.action.upper()} (conf={result.confidence:.1%}, {result.latency_ms:.0f}ms)")

## 6️⃣ Statistics & Cleanup

In [ ]:
# Get session statistics
stats = orchestrator.get_stats()

print("Session Statistics")
print("="*40)
print(f"Total Decisions: {stats['total_decisions']}")
print(f"Perception Calls: {stats['perception_calls']}")
print(f"Solver Calls: {stats['solver_calls']}")
print(f"Avg Latency: {stats['avg_latency_ms']:.0f}ms")

In [ ]:
# Cleanup resources
orchestrator.cleanup()

# Check VRAM
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_used:.2f} / {vram_total:.1f} GB")

print("✅ Resources released")